# Laser Head Camera Autofocus

This notebook uses the Poetry-managed environment. All dependencies from `pyproject.toml` are available.


In [ ]:
# Verify we're using the correct environment
import sys
print(f"Python: {sys.executable}")


In [ ]:
from femtum_sdk.api.contracts.v1.hardware.health_pb2 import HealthRequest
from femtum_sdk.api import HardwareApi

api = HardwareApi(host="localhost:5210")
api.__enter__()
health_response = api.health.GetHealth(HealthRequest())
print(health_response)

## Utils functions

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

from femtum_sdk.api.contracts.v1.dto.autofocus_pb2 import AutofocusResult


def print_result(result: AutofocusResult):
    if result.HasField("Success"):
        print("Autofocus succeeded:")
        print(result.Success)
    else:
        print("Autofocus failed:")
        print(result.Failure)


def plot_focus_score(result: AutofocusResult):
    positions = [float(step.AbsolutePosition.Value) for step in result.Steps]
    focus_scores = [float(step.FocusScore) for step in result.Steps]

    fig = make_subplots(
        rows=1,
        cols=1,
        subplot_titles=("Autofocus step-by-step sweep",),
    )
    fig.add_trace(
        go.Scatter(
            x=positions,
            y=focus_scores,
            mode="markers",
            marker={"size": 7},
            line={"width": 2},
            name="Autofocus steps",
        ),
        row=1,
        col=1,
    )
    
    if result.HasField("Success"):
        selected_position = float(result.Success.SelectedStep.AbsolutePosition.Value)
        selected_score = float(result.Success.SelectedStep.FocusScore)
        fig.add_trace(
            go.Scatter(
                x=[selected_position],
                y=[selected_score],
                mode="markers",
                marker={
                    "size": 12,
                    "symbol": "star",
                    "color": "#d62728",
                    "line": {"width": 1, "color": "#8c1d18"},
                },
                name="Selected step",
            ),
            row=1,
            col=1,
        )
    fig.update_xaxes(title_text="Position (um)", row=1, col=1)
    fig.update_yaxes(title_text="Focus score", row=1, col=1)
    fig.update_layout(
        title_text="Autofocus execution details",
        template="plotly_white",
        height=650,
        width=1100,
        legend={
            "x": 0.02,
            "y": 0.98,
            "xanchor": "left",
            "yanchor": "top",
            "bgcolor": "rgba(255,255,255,0.85)",
        },
    )

    fig.show()

## Execute autofocus

In [ ]:
from femtum_sdk.api.contracts.v1.hardware.camera_autofocus_pb2 import ExecuteAutofocusRequest
from femtum_sdk.api.contracts.v1.dto.autofocus_pb2 import AutofocusOptions, AutofocusMovementStrategy, MidPointStrategy
from femtum_sdk.api.contracts.v1.dto.bounding_box_pb2 import BoundingBox2D
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import DistanceMeasureUnit, DistanceMeasure

result = api.laser_head_camera_autofocus.ExecuteAutofocus(ExecuteAutofocusRequest(
    Roi=BoundingBox2D(X=2000, Y=2000, Width=512, Height=512),
    Options=AutofocusOptions(
        MovementStrategy=AutofocusMovementStrategy(
            MidPointForward=MidPointStrategy(
                MaxMoveLength=DistanceMeasure(
                    Value=100, 
                    Unit=DistanceMeasureUnit.Microns
                )
            )
        ),
        StepSize=DistanceMeasure(
            Value=1, 
            Unit=DistanceMeasureUnit.Microns
        ),
    )
))

plot_focus_score(result)
print_result(result)


## Execute autofocus for recipe (exception when autofocus analysis finds issues)

In [ ]:
from femtum_sdk.api.contracts.v1.hardware.camera_autofocus_pb2 import ExecuteAutofocusForRecipeRequest
from femtum_sdk.api.contracts.v1.dto.autofocus_pb2 import AutofocusOptions, AutofocusMovementStrategy, MidPointStrategy
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import DistanceMeasureUnit, DistanceMeasure

result = api.laser_head_camera_autofocus.ExecuteAutofocusForRecipe(ExecuteAutofocusForRecipeRequest(
    RecipeId="19",
    Options=AutofocusOptions(
        MovementStrategy=AutofocusMovementStrategy(
            MidPointForward=MidPointStrategy(
                MaxMoveLength=DistanceMeasure(
                    Value=25, 
                    Unit=DistanceMeasureUnit.Microns
                )
            )
        ),
        StepSize=DistanceMeasure(
            Value=1, 
            Unit=DistanceMeasureUnit.Microns
        ),
    )
))

plot_focus_score(result)
print_result(result)


## Execute autofocus for recipe (no exception when autofocus analysis finds issues)

In [ ]:
from femtum_sdk.api.contracts.v1.hardware.camera_autofocus_pb2 import ExecuteAutofocusForRecipeRequest
from femtum_sdk.api.contracts.v1.dto.autofocus_pb2 import AutofocusOptions, AutofocusMovementStrategy, MidPointStrategy
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import DistanceMeasureUnit, DistanceMeasure

result = api.laser_head_camera_autofocus.ExecuteAutofocusForRecipeIgnoreFailure(ExecuteAutofocusForRecipeRequest(
    RecipeId="19",
    Options=AutofocusOptions(
        MovementStrategy=AutofocusMovementStrategy(
            MidPointForward=MidPointStrategy(
                MaxMoveLength=DistanceMeasure(
                    Value=100, 
                    Unit=DistanceMeasureUnit.Microns
                )
            )
        ),
        StepSize=DistanceMeasure(
            Value=1, 
            Unit=DistanceMeasureUnit.Microns
        ),
    )
))

plot_focus_score(result)
print_result(result)